In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")   # ajuste se necessário

t = {p.stem.replace("olist_", "").replace("_dataset", ""): pd.read_csv(p)
     for p in RAW.glob("*.csv")}

for nome, df in t.items():
    print(nome, df.shape)
    print(list(df.columns), "\n")

customers (99441, 5)
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state'] 

geolocation (1000163, 5)
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state'] 

orders (99441, 8)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'] 

order_items (112650, 7)
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value'] 

order_payments (103886, 5)
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value'] 

order_reviews (99224, 7)
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp'] 

products (32951, 9)
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_

In [2]:
c, o = t["customers"], t["orders"]
print(c["customer_id"].nunique(), c["customer_unique_id"].nunique())
print(o["order_id"].is_unique)

base = o.merge(c, on="customer_id")
print((base.groupby("customer_unique_id").size() > 1).mean())

99441 96096
True
0.031187562437562436


In [3]:
print(t["order_items"].groupby("order_id").size().describe())
print(t["order_payments"].groupby("order_id").size().describe())
print((t["order_reviews"].groupby("order_id").size() > 1).sum())

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64
count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64
547


In [7]:
itens = t["order_items"]
certo = itens["price"].sum()
errado = itens.merge(t["order_payments"], on="order_id")["price"].sum()
print(certo, errado)

13591643.7 14209115.34


In [ ]:
o["order_purchase_timestamp"] = pd.to_datetime(o["order_purchase_timestamp"])
print(o.groupby(o["order_purchase_timestamp"].dt.to_period("M")).size())
print(o["order_purchase_timestamp"].max())


order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64
2018-10-17 17:30:18


In [8]:
b = base.copy()
b["order_purchase_timestamp"] = pd.to_datetime(b["order_purchase_timestamp"])
b = b.sort_values(["customer_unique_id", "order_purchase_timestamp"])
b["n"] = b.groupby("customer_unique_id").cumcount()

primeiro = b[b["n"] == 0].set_index("customer_unique_id")["order_purchase_timestamp"]
segundo = b[b["n"] == 1].set_index("customer_unique_id")["order_purchase_timestamp"]
dias = (segundo - primeiro.loc[segundo.index]).dt.days

print(dias.describe())
print("mesmo dia:", (dias == 0).mean())
print("até 180 dias:", (dias <= 180).mean())

count    2997.000000
mean       79.990657
std       110.088130
min         0.000000
25%         0.000000
50%        27.000000
75%       123.000000
max       608.000000
Name: order_purchase_timestamp, dtype: float64
mesmo dia: 0.30930930930930933
até 180 dias: 0.8301634968301635


In [9]:
b["data"] = b["order_purchase_timestamp"].dt.normalize()
primeira = b.groupby("customer_unique_id")["data"].transform("min")
b["dias_desde_primeiro"] = (b["data"] - primeira).dt.days

total = b["customer_unique_id"].nunique()
rec = b.loc[b["dias_desde_primeiro"] > 0, "customer_unique_id"].nunique()
rec180 = b.loc[b["dias_desde_primeiro"].between(1, 180), "customer_unique_id"].nunique()
print(total, rec, rec180, rec / total)

96096 2149 1634 0.022363053613053612


In [10]:
o["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [11]:
sem_av = ~t["orders"]["order_id"].isin(t["order_reviews"]["order_id"])
print(sem_av.sum(), sem_av.mean())

768 0.0077231725344676745


In [14]:
FIM = pd.Timestamp("2018-08-31 16:13:44")
LIMITE_PRIMEIRO_PEDIDO = FIM - pd.Timedelta(days=180)
print(FIM, LIMITE_PRIMEIRO_PEDIDO)

2018-08-31 16:13:44 2018-03-04 16:13:44
